# ARGUS Phase 3 — Prioritization Engine Exploration

This notebook demonstrates the Phase 3 Prioritization Engine for ARGUS:
1. **Catalogue & Configuration**: Loading fix catalogue and configuration parameters.
2. **Sample Attack Graphs**: Creating closed-form validation instances (Exercise 1 & Parallel Paths).
3. **Method Comparison**: Evaluating Naive Greedy, Stackelberg Security Game, and Shortest-Path Interdiction MILP.
4. **Budget Sweep & Efficient Frontier**: Analyzing residual attack probability vs. remediation budget.
5. **Synthetic Enterprise Demonstration**: Running on multi-stage IAM/Network hybrid attack graphs.

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

from prioritize.findings import load_catalogue, load_config, enumerate_findings
from prioritize.interdiction import build_and_solve
from prioritize.naive import naive_select
from prioritize.stackelberg_baseline import solve_stackelberg, round_to_binary
from prioritize.evaluate import attack_length, residual_prob, budget_sweep, compare_methods
from prioritize.tests.fixtures.graphs import exercise1, parallel_paths
from prioritize.benchmark import synthetic_enterprise

print('All Phase 3 modules loaded successfully!')

## 1. Fix Catalogue & Configuration

The catalogue maps security findings to their remediation deltas and costs.

In [ ]:
cat = load_catalogue()
print(f"Catalogue delta_max: {cat['delta_max']}")
for name, fix in cat['fixes'].items():
    print(f"  {name:<25}: cost={fix['cost']}, delta={fix['delta']}, scope={fix['scope']}")

## 2. Hand Example: Parallel Paths Dilemma

In this graph, two parallel paths exist into crown jewels. Naive greedy fixes only one path, leaving residual risk high (0.25). Interdiction MILP breaks both paths, drastically suppressing the attacker.

In [ ]:
G, entry, crown, findings = parallel_paths()
print(f'Graph: {len(G.nodes())} nodes, {len(G.edges())} edges, {len(findings)} findings')

# Budget = 2: compare Naive vs Interdiction
comp = compare_methods(G, entry, crown, findings, budget=2)
for m in comp['methods']:
    print(f"{m['name']:<20}: selected={m['selected']}, residual_p={m['residual_attack_prob']:.4f}")

## 3. Budget Sweep & Efficient Frontier

Sweeping remediation budget from 0 to 6 to visualize how residual risk drops.

In [ ]:
sweep = budget_sweep(G, entry, crown, findings, budgets=range(0, 7))
for pt in sweep:
    print(f"Budget {pt['budget']}: obj={pt['objective']:.4f}, residual_p={pt['residual_p']:.4e}, fixes={pt['selected']}")

## 4. Synthetic Enterprise Benchmark

Testing on a realistic enterprise graph with IAM privilege escalation and network ingress paths.

In [ ]:
G_ent, entry_ent, crown_ent, findings_ent = synthetic_enterprise(seed=42)
print(f'Enterprise Graph: {len(G_ent.nodes())} nodes, {len(G_ent.edges())} edges, {len(findings_ent)} findings')

ent_comp = compare_methods(G_ent, entry_ent, crown_ent, findings_ent, budget=4)
for m in ent_comp['methods']:
    print(f"{m['name']:<20}: fixes={len(m['selected'])}, residual_p={m['residual_attack_prob']:.4e}")